# EcoConsensus: fine-tune IndicBERT v2 with LoRA (Colab, self-contained)

Trains **ai4bharat/IndicBERTv2-MLM-only** for stance toward the fixed claim (*"Climate change and global warming is a serious concern."*),
labels **0 = favour, 1 = neutral, 2 = against**, sentence-pair input `(claim, comment)`, max length 128.
This notebook is self-contained (it imports nothing from the repo). The other model has its own notebook and can
train **at the same time on another Colab account**.

**Before you run**
1. On your Mac: `python -m src.training.prepare_data` → upload `data/ecoconsensus_data.zip` to `MyDrive/EcoConsensus/`.
2. Runtime → Change runtime type → **T4 GPU**.
3. Runtime → **Run all**. Only cell 1 (CONFIG) is meant to be edited.

**What it does:** LoRA (r 16, alpha 32, dropout 0.1, query+value, lr 3e-4) on `train_base` (already cleaned by
`prepare_data.py`; no cleaning happens here), class-weighted loss, AdamW, warmup + linear decay, fp16, early stopping on
`val` Macro-F1 (patience 2, up to 10 epochs). A checkpoint is written to Drive after **every epoch**: if Colab
disconnects, reconnect and **Run all** again; it resumes from the last finished epoch.
After training it writes the class probabilities for `meta`, `val` and `semeval_test` (no metrics are computed on
`semeval_test`), saves the adapter + `run_info.json`, logs to MLflow, and zips everything into
`MyDrive/EcoConsensus/outputs_indicbert/indicbert_results.zip`.

**After it finishes:** download `indicbert_results.zip` and unzip it **in the project root** on your Mac:
`unzip -o indicbert_results.zip -d /path/to/EcoConsensus`. It contains `models/finetuned/indicbert_lora/`,
`outputs/predictions/probs_indicbert_<split>.csv`, `outputs/metrics/mlflow_colab_indicbert.db` and the training log.

## 1. Config

In [ ]:
# ============================ 1. CONFIG - the only cell you normally edit ============================
MODEL_KEY  = "indicbert"
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"
EXPERIMENT_NAME = "finetune_final"

# ---- Where things live ----
DRIVE_FOLDER   = "EcoConsensus"            # Colab: MyDrive/<DRIVE_FOLDER>/ holds the data zip; outputs go below it
DATA_ZIP_NAME  = "ecoconsensus_data.zip"   # made by `python -m src.training.prepare_data` on the Mac
LOCAL_DATA_DIR = None                      # local smoke test only: folder with train_base.csv etc. (e.g. "data/processed")
LOCAL_OUT_DIR  = None                      # local smoke test only: output folder

# ---- Environment ----
INSTALL_DEPS = True                        # pip install the pinned versions below (Colab already has torch)
PINNED = ["transformers==5.22.2", "peft==0.21.2", "accelerate==1.15.0", "mlflow==3.18.0"]   # tested in the experiment
ALLOW_CPU_ON_CLOUD = False                 # stop if Colab/Kaggle has no GPU instead of training on CPU for hours
USE_MLFLOW = True

# ---- LoRA (experiment report 2026-11-08) ----
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.1
LORA_TARGETS = ["query", "value"]
LR = 3e-4

# ---- Training ----
SEED            = 42
MAX_EPOCHS      = 10
PATIENCE        = 2          # epochs without a better val Macro-F1 before stopping
BATCH_SIZE      = 16
EVAL_BATCH_SIZE = 64
MAX_LEN         = 128
WEIGHT_DECAY    = 0.01       # not on bias / LayerNorm
WARMUP_FRACTION = 0.1
GRAD_CLIP       = 1.0
HEAD_DROPOUT    = 0.1
FP16            = True       # mixed precision on CUDA (T4)

PREDICT_SPLITS  = ["meta", "val", "semeval_test"]   # probabilities for the stacking features
TRAIN_LIMIT     = None       # smoke test only: use the first N train_base rows (None = all)

## 2. Setup

In [ ]:
# ============================ 2. Install, imports, paths, device ============================
import os, sys, json, time, math, random, hashlib, platform, shutil, zipfile, subprocess, logging
import datetime as dt
from pathlib import Path
from importlib import metadata as importlib_metadata

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

if INSTALL_DEPS and (IN_COLAB or IN_KAGGLE):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED, "scikit-learn", "pandas"],
                       capture_output=True, text=True)
    print("pip:", "ok" if r.returncode == 0 else r.stderr[-1500:])

import numpy as np
import pandas as pd
import torch

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK_DIR = Path("/content")
elif IN_KAGGLE:
    BASE_DIR = Path("/kaggle/working")
    WORK_DIR = BASE_DIR
else:                                       # local smoke test: everything stays inside one output folder
    BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / f"outputs_{MODEL_KEY}" if (IN_COLAB or IN_KAGGLE) else Path(LOCAL_OUT_DIR or f"outputs_{MODEL_KEY}").resolve()
if not (IN_COLAB or IN_KAGGLE):
    WORK_DIR = OUT_DIR
for sub in ("checkpoints", "predictions", "logs"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)

LOG_FILE = OUT_DIR / "logs" / f"train_{MODEL_KEY}_{dt.datetime.now():%Y%m%d_%H%M%S}.txt"
log = logging.getLogger(f"train_{MODEL_KEY}")
log.handlers.clear(); log.setLevel(logging.INFO); log.propagate = False
for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(LOG_FILE, encoding="utf-8")):
    h.setFormatter(logging.Formatter("%(asctime)s %(message)s", "%H:%M:%S")); log.addHandler(h)


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
if DEVICE.type == "cpu" and (IN_COLAB or IN_KAGGLE) and not ALLOW_CPU_ON_CLOUD:
    raise RuntimeError("No GPU. Colab: Runtime > Change runtime type > T4 GPU, then Run all again.")
USE_AMP = FP16 and DEVICE.type == "cuda"


def _ver(p):
    try:
        return importlib_metadata.version(p)
    except Exception:
        return None


ENV_INFO = dict(python=platform.python_version(), platform=platform.platform(),
                runtime="colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local", device=DEVICE.type,
                gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
                **{f"lib_{p}": _ver(p) for p in ("torch", "transformers", "peft", "accelerate", "mlflow",
                                                  "scikit-learn", "pandas")})
log.info("Device %s (fp16 %s) | outputs %s", DEVICE, USE_AMP, OUT_DIR)
log.info("%s", {k: v for k, v in ENV_INFO.items() if v})

## 3. Data (already cleaned by `prepare_data.py`)

In [ ]:
# ============================ 3. Data: unzip the bundle, load the cleaned splits, verify ============================
LABELS = ["favour", "neutral", "against"]          # ids 0, 1, 2 (fixed project mapping)
SPLIT_FILES = ["train_base", "meta", "val", "semeval_test"]


def sha256_16(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()[:16]


def locate_data() -> Path:
    if LOCAL_DATA_DIR and (Path(LOCAL_DATA_DIR) / "train_base.csv").exists():
        return Path(LOCAL_DATA_DIR)
    zips = []
    if IN_COLAB:
        zips.append(BASE_DIR / DATA_ZIP_NAME)
    if IN_KAGGLE:
        for p in Path("/kaggle/input").rglob("train_base.csv"):      # Kaggle auto-extracts uploaded zips
            return p.parent
        zips += list(Path("/kaggle/input").rglob(DATA_ZIP_NAME))
    zips += [Path(DATA_ZIP_NAME), Path("data") / DATA_ZIP_NAME]
    for z in zips:
        if z.exists():
            dest = WORK_DIR / "ecoconsensus_data"
            shutil.rmtree(dest, ignore_errors=True)                   # never mix files from an older bundle
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
            log.info("Unzipped %s -> %s", z, dest)
            return dest
    raise FileNotFoundError(f"{DATA_ZIP_NAME} not found (looked in {[str(z) for z in zips]}). "
                            f"Upload it to MyDrive/{DRIVE_FOLDER}/.")


DATA_DIR = locate_data()
MANIFEST = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8")) if (DATA_DIR / "manifest.json").exists() else {}
SPLITS = {}
for s in SPLIT_FILES:
    p = DATA_DIR / f"{s}.csv"
    want = MANIFEST.get("splits", {}).get(s, {}).get("sha256")
    if want and sha256_16(p) != want:
        raise ValueError(f"{p.name} does not match manifest.json (corrupted or mixed bundle). Re-upload the zip.")
    df = pd.read_csv(p, dtype={"id": str, "source_id": str, "group_id": str}, keep_default_na=False)
    missing = {"id", "source_id", "claim", "comment", "label"} - set(df.columns)
    if missing:
        raise ValueError(f"{p.name} lacks columns {missing}; rebuild the bundle with the current prepare_data.py")
    df["label"] = df["label"].astype(int)
    SPLITS[s] = df

for a in SPLIT_FILES:                                  # the bundle was checked when built; check again
    for b in SPLIT_FILES[SPLIT_FILES.index(a) + 1:]:
        both = set(SPLITS[a]["source_id"]) & set(SPLITS[b]["source_id"])
        assert not both, f"{len(both)} source ids in both {a} and {b}"
if TRAIN_LIMIT:
    SPLITS["train_base"] = SPLITS["train_base"].head(TRAIN_LIMIT)
DATA_HASH = hashlib.sha1("".join(sha256_16(DATA_DIR / f"{s}.csv") for s in SPLIT_FILES).encode()).hexdigest()[:12]
for s, d in SPLITS.items():
    log.info("%-13s %6d rows  %s", s, len(d), d["label"].value_counts().sort_index().to_dict())
log.info("bundle created %s | data hash %s", MANIFEST.get("created"), DATA_HASH)

## 4. Training code

In [ ]:
# ============================ 4. Training code: dataset, LoRA model, optimizer, prediction ============================
from contextlib import nullcontext
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    pr, rc, f, s = precision_recall_fscore_support(y, p, labels=[0, 1, 2], zero_division=0)
    return dict(n=int(len(y)), macro_f1=float(f1_score(y, p, labels=[0, 1, 2], average="macro", zero_division=0)),
                accuracy=float(accuracy_score(y, p)),
                per_class={LABELS[i]: dict(precision=float(pr[i]), recall=float(rc[i]), f1=float(f[i]),
                                           support=int(s[i])) for i in range(3)},
                confusion=confusion_matrix(y, p, labels=[0, 1, 2]).tolist())


class PairDataset(torch.utils.data.Dataset):
    """Pre-tokenised (claim, comment) pairs; the text is already cleaned by prepare_data.py."""
    def __init__(self, df, tok):
        enc = tok(df["claim"].tolist(), df["comment"].tolist(), truncation=True, max_length=MAX_LEN)
        self.keys = [k for k in ("input_ids", "attention_mask", "token_type_ids") if k in enc]
        self.enc = {k: enc[k] for k in self.keys}
        self.y = df["label"].to_numpy()
        self.lengths = [len(x) for x in self.enc["input_ids"]]

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.keys}
        item["labels"] = int(self.y[i])
        return item


def make_collate(pad_id):
    def collate(batch):
        width = max(len(b["input_ids"]) for b in batch)
        out = {}
        for k in batch[0]:
            if k == "labels":
                out[k] = torch.tensor([b[k] for b in batch], dtype=torch.long)
            else:
                fill = pad_id if k == "input_ids" else 0
                out[k] = torch.tensor([b[k] + [fill] * (width - len(b[k])) for b in batch], dtype=torch.long)
        return out
    return collate


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()


@torch.no_grad()
def predict_probs(model, ds):
    model.eval()
    collate = make_collate(PAD_ID)
    order = np.argsort(ds.lengths)                      # length-sorted batches: less padding
    probs = np.zeros((len(ds), 3), dtype=np.float32)
    for s in range(0, len(order), EVAL_BATCH_SIZE):
        idx = order[s:s + EVAL_BATCH_SIZE]
        batch = collate([ds[int(i)] for i in idx])
        batch.pop("labels")
        with amp_ctx():
            logits = model(**{k: v.to(DEVICE) for k, v in batch.items()}).logits
        probs[idx] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    return probs


def build_model():
    kw = dict(num_labels=3, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)},
              ignore_mismatched_sizes=True)
    if DEVICE.type == "mps":
        kw["attn_implementation"] = "eager"
    try:
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, classifier_dropout=HEAD_DROPOUT, **kw)
    except TypeError:
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, **kw)
    # the new classifier (and the pooler, which may be untrained in MLM-only checkpoints) are trained in full
    head = ["classifier"] + (["pooler"] if getattr(model.base_model, "pooler", None) is not None else [])
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     target_modules=LORA_TARGETS, modules_to_save=head)
    model = get_peft_model(model, cfg).to(DEVICE)
    return model


def make_optimizer(model):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if p.requires_grad:
            (no_decay if n.endswith(".bias") or "layernorm" in n.lower() or "layer_norm" in n.lower() else decay).append(p)
    return torch.optim.AdamW([dict(params=decay, weight_decay=WEIGHT_DECAY), dict(params=no_decay, weight_decay=0.0)],
                             lr=LR)


def trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable(model, state):
    with torch.no_grad():
        params = dict(model.named_parameters())
        for n, t in state.items():
            params[n].copy_(t.to(params[n].device))


def atomic_save(obj, p):
    tmp = Path(str(p) + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(p)


set_seed(SEED)
TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME)
PAD_ID = TOKENIZER.pad_token_id if TOKENIZER.pad_token_id is not None else 0
DATASETS = {s: PairDataset(df, TOKENIZER) for s, df in SPLITS.items() if s == "train_base" or s in PREDICT_SPLITS or s == "val"}
lens = np.array(DATASETS["train_base"].lengths)
log.info("train_base tokens: median %d, p99 %d, max %d, truncated at %d: %d rows", np.median(lens),
         np.percentile(lens, 99), lens.max(), MAX_LEN, int((lens >= MAX_LEN).sum()))

## 5. Train (resumes after a disconnect: just Run all again)

In [ ]:
# ============================ 5. Train (resumable: a checkpoint per epoch on Drive) ============================
RUN_CONFIG = dict(model=MODEL_NAME, method="lora", r=LORA_R, alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                  targets=LORA_TARGETS, lr=LR, seed=SEED, max_epochs=MAX_EPOCHS, patience=PATIENCE, bs=BATCH_SIZE,
                  max_len=MAX_LEN, wd=WEIGHT_DECAY, warmup=WARMUP_FRACTION, clip=GRAD_CLIP, head_dropout=HEAD_DROPOUT,
                  amp=USE_AMP, data=DATA_HASH, train_limit=TRAIN_LIMIT)
RUN_HASH = hashlib.sha1(json.dumps(RUN_CONFIG, sort_keys=True).encode()).hexdigest()[:10]
RUN_NAME = f"{MODEL_KEY}__lora__seed{SEED}__{RUN_HASH}"
CKPT_DIR = OUT_DIR / "checkpoints" / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
STATE_PT, BEST_PT, DONE_JSON = CKPT_DIR / "state.pt", CKPT_DIR / "best_trainable.pt", CKPT_DIR / "done.json"
log.info("run %s", RUN_NAME)

model = build_model()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
log.info("trainable parameters %s of %s (%.2f%%)", f"{trainable:,}", f"{total:,}", 100 * trainable / total)

y_train = SPLITS["train_base"]["label"].to_numpy()
counts = np.bincount(y_train, minlength=3).astype(float)
CLASS_WEIGHTS = np.where(counts > 0, len(y_train) / (3.0 * np.maximum(counts, 1)), 1.0)
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(CLASS_WEIGHTS, dtype=torch.float32, device=DEVICE))
log.info("class weights %s", np.round(CLASS_WEIGHTS, 3).tolist())

steps_per_epoch = math.ceil(len(DATASETS["train_base"]) / BATCH_SIZE)
total_steps = steps_per_epoch * MAX_EPOCHS
warm = max(1, int(WARMUP_FRACTION * total_steps))
opt = make_optimizer(model)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: s / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
params = [p for p in model.parameters() if p.requires_grad]

history, best_score, best_epoch, since_best, start_epoch, train_sec = [], -1.0, 0, 0, 1, 0.0
if DONE_JSON.exists():
    log.info("this run already finished; loading its best weights (delete %s to retrain)", CKPT_DIR)
    st = torch.load(STATE_PT, map_location="cpu", weights_only=False)
    history, best_score, best_epoch, train_sec = st["history"], st["best_score"], st["best_epoch"], st["train_sec"]
    start_epoch = MAX_EPOCHS + 1
elif STATE_PT.exists():
    st = torch.load(STATE_PT, map_location="cpu", weights_only=False)
    load_trainable(model, st["trainable"])
    opt.load_state_dict(st["optimizer"]); sched.load_state_dict(st["scheduler"]); scaler.load_state_dict(st["scaler"])
    history, best_score, best_epoch, since_best = st["history"], st["best_score"], st["best_epoch"], st["since_best"]
    train_sec, start_epoch = st["train_sec"], st["epoch"] + 1
    random.setstate(st["rng_py"]); np.random.set_state(st["rng_np"]); torch.set_rng_state(st["rng_torch"])
    log.info("RESUMED after epoch %d (best val Macro-F1 %.4f at epoch %d)", st["epoch"], best_score, best_epoch)

collate = make_collate(PAD_ID)
for epoch in range(start_epoch, MAX_EPOCHS + 1):
    if since_best >= PATIENCE:
        break
    loader = torch.utils.data.DataLoader(DATASETS["train_base"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                                         generator=torch.Generator().manual_seed(SEED * 1000 + epoch))
    model.train()
    t0, loss_sum, nb = time.perf_counter(), 0.0, 0
    for batch in loader:
        labels = batch.pop("labels").to(DEVICE)
        with amp_ctx():
            logits = model(**{k: v.to(DEVICE) for k, v in batch.items()}).logits
        loss = loss_fn(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        scaler.step(opt); scaler.update(); sched.step()
        loss_sum += loss.item(); nb += 1
    ep_sec = time.perf_counter() - t0
    train_sec += ep_sec
    vm = metrics(SPLITS["val"]["label"], predict_probs(model, DATASETS["val"]).argmax(1))
    row = dict(epoch=epoch, train_loss=loss_sum / max(1, nb), val_macro_f1=vm["macro_f1"], val_accuracy=vm["accuracy"],
               val_f1_against=vm["per_class"]["against"]["f1"], epoch_sec=round(ep_sec, 1), lr=sched.get_last_lr()[0])
    history.append(row)
    improved = vm["macro_f1"] > best_score
    if improved:
        best_score, best_epoch, since_best = vm["macro_f1"], epoch, 0
        atomic_save(trainable_state(model), BEST_PT)
    else:
        since_best += 1
    atomic_save(dict(epoch=epoch, trainable=trainable_state(model), optimizer=opt.state_dict(),
                     scheduler=sched.state_dict(), scaler=scaler.state_dict(), history=history, best_score=best_score,
                     best_epoch=best_epoch, since_best=since_best, train_sec=train_sec, run_config=RUN_CONFIG,
                     rng_py=random.getstate(), rng_np=np.random.get_state(), rng_torch=torch.get_rng_state()), STATE_PT)
    log.info("epoch %2d | loss %.4f | val Macro-F1 %.4f%s | acc %.3f | F1 against %.3f | %.0fs", epoch, row["train_loss"],
             vm["macro_f1"], " *" if improved else "", vm["accuracy"], row["val_f1_against"], ep_sec)
    if since_best >= PATIENCE:
        log.info("early stopping (no improvement for %d epochs)", PATIENCE)
        break

(DONE_JSON).write_text(json.dumps(dict(best_epoch=best_epoch, best_val_macro_f1=best_score,
                                       finished=dt.datetime.now().isoformat(timespec="seconds"))), encoding="utf-8")
load_trainable(model, torch.load(BEST_PT, map_location="cpu", weights_only=False))
log.info("restored best epoch %d (val Macro-F1 %.4f); training time %.1f min", best_epoch, best_score, train_sec / 60)

## 6. Save, predict, zip the results

In [ ]:
# ============================ 6. Save adapter, predict meta / val / semeval_test, MLflow, results zip ============================
ADAPTER_DIR = OUT_DIR / f"{MODEL_KEY}_lora"
if ADAPTER_DIR.exists():
    shutil.rmtree(ADAPTER_DIR)
model.save_pretrained(ADAPTER_DIR)
TOKENIZER.save_pretrained(ADAPTER_DIR)

try:
    from huggingface_hub import HfApi
    BASE_REVISION = HfApi().model_info(MODEL_NAME).sha
except Exception:
    BASE_REVISION = None

final = {}
pred_files = []
for s in PREDICT_SPLITS:
    pr = predict_probs(model, DATASETS[s])
    df = SPLITS[s]
    out = df[["id", "source_id", "language", "variant", "script_group", "label"]].copy()
    for j in range(3):
        out[f"{MODEL_KEY}_p{j}"] = pr[:, j].round(6)
    out[f"{MODEL_KEY}_pred"] = pr.argmax(1)
    p = OUT_DIR / "predictions" / f"probs_{MODEL_KEY}_{s}.csv"
    out.to_csv(p, index=False)
    pred_files.append(p)
    if s != "semeval_test":                    # SemEval test is scored once, at the very end (test.py)
        m = metrics(df["label"], pr.argmax(1))
        m["by_variant"] = {lv: float(f1_score(g["label"], pr[g.index].argmax(1), labels=[0, 1, 2], average="macro",
                                                zero_division=0))
                           for lv, g in df.assign(lv=df["language"] + "/" + df["variant"]).groupby("lv")}
        final[s] = m
        log.info("%-5s Macro-F1 %.4f | acc %.4f | F1 favour %.3f neutral %.3f against %.3f", s, m["macro_f1"],
                 m["accuracy"], *[m["per_class"][l]["f1"] for l in LABELS])
    log.info("wrote %s (%d rows)", p.name, len(out))

RUN_INFO = dict(component=MODEL_KEY, run_name=RUN_NAME, base_model=MODEL_NAME, base_revision=BASE_REVISION,
                method="lora", config=RUN_CONFIG, labels={i: l for i, l in enumerate(LABELS)},
                input="sentence pair (claim, comment), cleaned by src/preprocessing.py", max_length=MAX_LEN,
                best_epoch=best_epoch, epochs_run=len(history), history=history, metrics=final,
                trainable_params=trainable, total_params=total, class_weights=CLASS_WEIGHTS.tolist(),
                train_rows=len(SPLITS["train_base"]), train_time_sec=round(train_sec, 1), data_hash=DATA_HASH,
                bundle_created=MANIFEST.get("created"), env=ENV_INFO,
                finished=dt.datetime.now().isoformat(timespec="seconds"), mlflow_run_id=None)

MLFLOW_DB = WORK_DIR / f"mlflow_colab_{MODEL_KEY}.db"
if USE_MLFLOW:
    try:
        import mlflow
        if (OUT_DIR / MLFLOW_DB.name).exists() and not MLFLOW_DB.exists():
            shutil.copy2(OUT_DIR / MLFLOW_DB.name, MLFLOW_DB)
        mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB}")
        if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
            mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=(WORK_DIR / "mlartifacts").as_uri())
        mlflow.set_experiment(EXPERIMENT_NAME)
        with mlflow.start_run(run_name=RUN_NAME) as run:
            mlflow.log_params({k: str(v) for k, v in RUN_CONFIG.items()})
            mlflow.log_params(dict(trainable_params=trainable, best_epoch=best_epoch, base_revision=BASE_REVISION))
            for h in history:
                for k in ("train_loss", "val_macro_f1", "val_accuracy", "val_f1_against"):
                    mlflow.log_metric(k, h[k], step=h["epoch"])
            for s, m in final.items():
                mlflow.log_metric(f"final_{s}_macro_f1", m["macro_f1"])
                mlflow.log_metric(f"final_{s}_accuracy", m["accuracy"])
            mlflow.log_metric("train_time_sec", train_sec)
            RUN_INFO["mlflow_run_id"] = run.info.run_id
            (ADAPTER_DIR / "run_info.json").write_text(json.dumps(RUN_INFO, indent=2, default=str), encoding="utf-8")
            mlflow.log_artifact(str(ADAPTER_DIR / "run_info.json"))
        if MLFLOW_DB.resolve() != (OUT_DIR / MLFLOW_DB.name).resolve():
            shutil.copy2(MLFLOW_DB, OUT_DIR / MLFLOW_DB.name)          # Colab: /content -> Drive
    except Exception as e:
        log.warning("MLflow failed (training results are still saved): %s: %s", type(e).__name__, str(e)[:200])
(ADAPTER_DIR / "run_info.json").write_text(json.dumps(RUN_INFO, indent=2, default=str), encoding="utf-8")

# results zip with project-relative paths: unzip it in the project root on the Mac
ZIP = OUT_DIR / f"{MODEL_KEY}_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(ADAPTER_DIR.rglob("*")):
        if f.is_file():
            zf.write(f, f"models/finetuned/{MODEL_KEY}_lora/{f.relative_to(ADAPTER_DIR)}")
    for f in pred_files:
        zf.write(f, f"outputs/predictions/{f.name}")
    if (OUT_DIR / MLFLOW_DB.name).exists():
        zf.write(OUT_DIR / MLFLOW_DB.name, f"outputs/metrics/{MLFLOW_DB.name}")
    for f in (OUT_DIR / "logs").glob("*.txt"):
        zf.write(f, f"outputs/logs/colab_{f.name}")
log.info("DONE. Download %s and run on the Mac:  unzip -o %s_results.zip -d /path/to/EcoConsensus", ZIP, MODEL_KEY)